# 2. Train the room-style classifier (MobileNetV3-Small)
Open in Google Colab -> **T4 GPU**. You need a free Kaggle account: kaggle.com -> Settings -> Create New Token (downloads `kaggle.json`), then upload it when the cell asks.
Dataset: https://www.kaggle.com/datasets/stepanyarullin/interior-design-styles (one folder per style).
Outputs `style_mobilenetv3.pt` and `style_classes.json`. Put both in `backend/perception/weights/`.
Check the licence on the dataset page before publishing anything.

In [ ]:
!pip -q install kaggle

In [ ]:
from google.colab import files
files.upload()                                   # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d stepanyarullin/interior-design-styles -p data --unzip

In [ ]:
import os
# find the folder whose sub-folders are the style classes (each holds images)
def has_imgs(d): return any(f.lower().endswith(('.jpg','.jpeg','.png')) for f in os.listdir(d))
root = None
for dp, dn, fn in os.walk("data"):
    subs = [d for d in dn if has_imgs(os.path.join(dp, d))]
    if len(subs) >= 3: root = dp; break
assert root, "Could not find class folders - look inside the data folder and set root by hand"
print("Using", root, sorted(os.listdir(root)))

In [ ]:
import torch, json, torch.nn as nn
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from torch.utils.data import DataLoader, random_split

norm = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
train_tf = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.6,1)), transforms.RandomHorizontalFlip(),
                               transforms.ColorJitter(0.2,0.2,0.2), transforms.ToTensor(), norm])
val_tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), norm])
full = datasets.ImageFolder(root)
n_val = int(0.2*len(full)); g = torch.Generator().manual_seed(0)
tr, va = random_split(range(len(full)), [len(full)-n_val, n_val], generator=g)
class Sub(torch.utils.data.Dataset):
    def __init__(s, idx, tf): s.idx, s.tf = list(idx), tf
    def __len__(s): return len(s.idx)
    def __getitem__(s, i):
        path, y = full.samples[s.idx[i]]
        return s.tf(full.loader(path)), y
tl = DataLoader(Sub(tr, train_tf), batch_size=32, shuffle=True, num_workers=2)
vl = DataLoader(Sub(va, val_tf), batch_size=64, num_workers=2)
classes = full.classes; print(len(classes), "styles:", classes)

In [ ]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(classes))
model.to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
lossf = nn.CrossEntropyLoss(label_smoothing=0.1)
best = 0
for ep in range(12):
    model.train()
    for x, y in tl:
        x, y = x.to(dev), y.to(dev); opt.zero_grad(); lossf(model(x), y).backward(); opt.step()
    model.eval(); ok = tot = 0
    with torch.no_grad():
        for x, y in vl:
            ok += (model(x.to(dev)).argmax(1).cpu() == y).sum().item(); tot += len(y)
    acc = ok / tot; print(f"epoch {ep+1}: validation accuracy {acc:.3f}")
    if acc > best: best = acc; torch.save(model.state_dict(), "style_mobilenetv3.pt")
json.dump(classes, open("style_classes.json", "w"))
print("best validation accuracy:", round(best, 3))

In [ ]:
from google.colab import files
files.download("style_mobilenetv3.pt"); files.download("style_classes.json")